# Comparing Checkov built-in vs custom policies for Kubernetes security enforcement

Checkov ships roughly 1,400 built-in policies across AWS/Azure/GCP (with CIS/PCI/HIPAA/SOC2 mappings), but for Kubernetes security enforcement teams frequently need checks the built-ins don't cover. This notebook compares running built-in checks against writing a custom Python check, and shows where each approach pays off.

## Purpose

Understand when Checkov's built-in Kubernetes policies are enough, and when you should write a custom check. The notebook covers:
- Running built-in checks against a sample Kubernetes manifest
- Writing a small custom check that enforces a rule the built-ins don't
- Comparing coverage and maintenance cost between the two approaches

## Prerequisites

- Python 3.8+
- `checkov` installed (`pip install checkov`)
- A sample Kubernetes manifest with intentional misconfigurations

In [ ]:
import subprocess
import json
import tempfile
from pathlib import Path

def check_prerequisites():
    """Verify checkov is accessible."""
    result = subprocess.run(["which", "checkov"], capture_output=True, text=True)
    if result.returncode != 0:
        print("WARNING: checkov not found in PATH — cells will fail")
    else:
        print(f"OK: checkov found at {result.stdout.strip()}")

check_prerequisites()

## Step 1: Create a sample Kubernetes manifest

A pod that runs as root with a writable root filesystem and privileged escalation — exactly the kind of thing a security gate should catch.

In [ ]:
SAMPLE_MANIFEST = '''
apiVersion: v1
kind: Pod
metadata:
  name: risky-app
spec:
  containers:
    - name: app
      image: nginx:latest
      securityContext:
        runAsUser: 0
        privileged: true
        readOnlyRootFilesystem: false
---
apiVersion: v1
kind: Pod
metadata:
  name: ok-app
spec:
  containers:
    - name: app
      image: nginx:latest
      securityContext:
        runAsUser: 1000
        privileged: false
        readOnlyRootFilesystem: true
'''

k8s_dir = Path(tempfile.mkdtemp(prefix="checkov_k8s_"))
manifest = k8s_dir / "pod.yaml"
manifest.write_text(SAMPLE_MANIFEST)
print(f"Kubernetes manifest created at: {manifest}")

## Step 2: Run built-in checks

Checkov's built-in Kubernetes suite covers many pod-security concerns (privileged containers, host namespaces, etc.). Point it at the manifest.

In [ ]:
def run_builtin_scan(manifest_path: Path) -> dict:
    """Run Checkov built-in Kubernetes scan on a manifest.

    Returns parsed JSON output.
    """
    result = subprocess.run(
        ["checkov", "--file", str(manifest_path),
         "--framework", "kubernetes",
         "--output", "json", "--compact"],
        capture_output=True, text=True
    )
    if result.returncode not in (0, 1):
        print(f"ERROR: Checkov exited with code {result.returncode}")
        print(result.stderr[:500])
        return {"results": {}}
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e}")
        return {"results": {}}


builtin_results = run_builtin_scan(manifest)

In [ ]:
# Extract failed check IDs from built-in scan
builtin_failed = []
for entry in builtin_results.get("results", {}).get("failed_checks", []):
    builtin_failed.append(entry.get("check_id", "?"))

print(f"Built-in failed checks: {len(builtin_failed)}")
for cid in sorted(set(builtin_failed)):
    print(f"  - {cid}")

## Step 3: Write a custom check

Suppose your org requires every pod to set `runAsNonRoot: true` explicitly — a rule the built-in policies don't enforce by default. Custom checks extend `BaseK8Check` and implement `scan_resource_conf`.

In [ ]:
# checkov_custom_checks/run_as_nonroot.py
CUSTOM_CHECK = '''
import checkov.common.models.enums as enums
from checkov.kubernetes.checks.resource.base_spec_check import BaseK8Check


class RunAsNonRootRequired(BaseK8Check):
    def __init__(self):
        name = "Pod must set runAsNonRoot explicitly"
        id = "CKV_CUSTOM_K8S_001"
        supported_kind = ["Pod"]
        categories = [enums.CheckCategories.KUBERNETES]
        super().__init__(name=name, id=id, categories=categories,
                         supported_entities=supported_kind)

    def scan_resource_conf(self, conf):
        spec = conf.get("spec", {})
        containers = spec.get("containers", [])
        for c in containers:
            sc = c.get("securityContext", {})
            if sc.get("runAsNonRoot") is not True:
                return enums.CheckResult.FAILED
        return enums.CheckResult.PASSED
'''

custom_dir = k8s_dir / "checkov_custom_checks"
custom_dir.mkdir()
(custom_dir / "run_as_nonroot.py").write_text(CUSTOM_CHECK)
print(f"Custom check written to {custom_dir / 'run_as_nonroot.py'}")

## Step 4: Run with the custom check

Load the custom check via `--external-checks-dir`. Checkov will merge the custom check results with its built-in checks.

In [ ]:
def run_custom_scan(manifest_path: Path, checks_dir: Path) -> dict:
    """Run Checkov with a custom check directory loaded."""
    result = subprocess.run(
        ["checkov", "--file", str(manifest_path),
         "--framework", "kubernetes",
         "--external-checks-dir", str(checks_dir),
         "--output", "json", "--compact"],
        capture_output=True, text=True
    )
    if result.returncode not in (0, 1):
        print(f"ERROR: Checkov exited with code {result.returncode}")
        print(result.stderr[:500])
        return {"results": {}}
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e}")
        return {"results": {}}


custom_results = run_custom_scan(manifest, custom_dir)
custom_failed = [e.get("check_id", "?")
                 for e in custom_results.get("results", {}).get("failed_checks", [])]
print(f"Custom-scan failed checks: {len(custom_failed)}")
for cid in sorted(set(custom_failed)):
    print(f"  - {cid}")

## Summary

**Built-in policies** give broad, maintained, standards-mapped coverage for free. For Kubernetes they already flag privileged containers and similar issues, so most teams start here and only add custom checks for org-specific rules the built-ins don't express.

**Custom checks** fill the gaps — your own naming conventions, internal guardrails, or a stricter baseline than the defaults. The maintenance cost is yours: a custom check is code you own, test, and keep current as the API evolves.

**Recommendation:** lean on built-ins for breadth, write custom checks only for rules unique to your org.

In [ ]:
# Clean up temp directory
import shutil
shutil.rmtree(k8s_dir, ignore_errors=True)
print(f"Cleaned up {k8s_dir}")